# Lean-RGC v0.2 Quickstart

This notebook runs the automation stack in dry-run mode first, then shows the commands for real Lean/Lake mode.

In [ ]:
from pathlib import Path
import os, json, subprocess, textwrap, sys
ROOT = Path("/content/lean_rgc_v02")
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)

## 1. Upload / unzip package
Upload `lean_rgc_automation_stack_v02.zip` to Colab, or mount it under `/content`.

In [ ]:
ZIP = Path("/content/lean_rgc_automation_stack_v02.zip")
if not ZIP.exists():
    print("Upload lean_rgc_automation_stack_v02.zip, then re-run this cell.")
else:
    subprocess.run(["bash", "-lc", f"rm -rf {ROOT}/stack && mkdir -p {ROOT}/stack && unzip -qo {ZIP} -d {ROOT}/stack"], check=True)
    STACK = next((ROOT/"stack").glob("lean_rgc_stack*"), ROOT/"stack"/"lean_rgc_stack")
    print("STACK=", STACK)

In [ ]:
import sys, os
STACK = next((ROOT/"stack").glob("lean_rgc_stack*"), ROOT/"stack"/"lean_rgc_stack")
os.chdir(STACK)
sys.path.insert(0, str(STACK))
!python -m lean_rgc.cli --help

## 2. Dry-run batch audit

In [ ]:
RUN = ROOT / "runs" / "dry_batch"
!python -m lean_rgc.cli batch-audit \
  --tasks examples/minimal_theorems.jsonl \
  --actions examples/tactic_templates.jsonl \
  --out {RUN} \
  --dry-run \
  --jobs 4 \
  --max-actions 8

## 3. Response quotient, response learner, carrier coker

In [ ]:
!python -m lean_rgc.cli quotient --responses {RUN}/responses.jsonl --out {RUN}/components.jsonl
!python -m lean_rgc.cli train-response --responses {RUN}/responses.jsonl --actions examples/tactic_templates.jsonl --out {RUN}/response_model.json
!python -m lean_rgc.cli predict-response --model {RUN}/response_model.json --actions examples/tactic_templates.jsonl --out {RUN}/preds.jsonl
!python -m lean_rgc.cli carrier-coker --defects {RUN}/defects.jsonl --actions examples/tactic_templates.jsonl --out {RUN}/carrier_coker.json
!python -m lean_rgc.cli carrier-generate --defects {RUN}/defects.jsonl --out {RUN}/carrier_gen.jsonl

## 4. Inspect results

In [ ]:
import pandas as pd, json
for name in ["summary.json", "carrier_coker.json"]:
    print("\n==", name, "==")
    print(Path(RUN/name).read_text()[:2000])
components = [json.loads(l) for l in Path(RUN/"components.jsonl").read_text().splitlines() if l.strip()]
pd.DataFrame(components).head()

## 5. Optional: create a Lake template

In [ ]:
!python -m lean_rgc.cli init-lake --out {ROOT}/lean_playground --no-mathlib
!find {ROOT}/lean_playground -maxdepth 3 -type f -print

## 6. Real Lean mode
After installing Lean/Lake and building a project, remove `--dry-run` and set `--workdir` to the Lake project root.

In [ ]:
# Example only; requires Lean/Lake in the runtime.
# !python -m lean_rgc.cli batch-audit \
#   --tasks examples/minimal_theorems.jsonl \
#   --actions examples/tactic_templates.jsonl \
#   --out {ROOT}/runs/real_audit \
#   --workdir {ROOT}/lean_playground \
#   --lean-cmd "lake env lean" \
#   --jobs 4